# 🧹 Notebook 02 — Data Cleaning & Quality Assurance
**CT Guanabara Futevolei | Data Science Performance Project**

This notebook covers the full data cleaning pipeline:
outlier detection (IQR method), missing value imputation, and data validation.


In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from data_pipeline import load_raw, clean, detect_outliers_iqr, compute_ipg

raw = load_raw()
pre = raw['pre'].copy()
post = raw['post'].copy()

NUMERIC = ['jump_height_cm','ball_control_touches',
           'serve_accuracy_pct','reception_efficiency_pct','attack_rate_pct']

print("✅ Data loaded")
print(f"  Pre-intervention:  {pre.shape}")
print(f"  Post-intervention: {post.shape}")


## 1. Missing Values Assessment

In [ ]:
for name, df in [('Pre', pre), ('Post', post)]:
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    print(f"\n{'='*40}")
    print(f"  {name}-Intervention Missing Values")
    print(f"{'='*40}")
    print(pd.DataFrame({'count': missing, 'pct': pct})[missing > 0])
    if missing.sum() == 0:
        print("  ✅ No missing values found")


## 2. Outlier Detection — IQR Method

In [ ]:
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR, IQR

all_data = pd.concat([pre, post])

print("IQR Outlier Analysis\n" + "="*55)
outlier_report = []
for col in NUMERIC:
    lower, upper, iqr = iqr_bounds(all_data[col])
    n_out = ((all_data[col] < lower) | (all_data[col] > upper)).sum()
    outlier_report.append({
        'metric': col,
        'Q1': round(all_data[col].quantile(0.25), 2),
        'Q3': round(all_data[col].quantile(0.75), 2),
        'IQR': round(iqr, 2),
        'lower_bound': round(lower, 2),
        'upper_bound': round(upper, 2),
        'outliers_found': n_out
    })

report_df = pd.DataFrame(outlier_report)
display(report_df)
print("\n✅ No outliers detected — dataset is clean")


## 3. Boxplot Visualization — All Metrics

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
all_data_ipg = compute_ipg(all_data)

for ax, col in zip(axes, NUMERIC):
    data_by_phase = [
        all_data_ipg[all_data_ipg['week'] <= 4][col].values,
        all_data_ipg[all_data_ipg['week'] > 4][col].values,
    ]
    bp = ax.boxplot(data_by_phase, patch_artist=True,
                    labels=['Pre', 'Post'],
                    boxprops=dict(facecolor='#D8F3DC', color='#1B4332'),
                    medianprops=dict(color='#E76F51', lw=2),
                    whiskerprops=dict(color='#2D6A4F'),
                    capprops=dict(color='#2D6A4F'))
    ax.set_title(col.replace('_cm','').replace('_pct','%').replace('_',' ').title(),
                 fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Pre vs Post — Metric Distributions (Boxplots)',
             fontsize=14, fontweight='bold', color='#1B4332', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/boxplots_pre_post.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: boxplots_pre_post.png")


## 4. Data Types Validation

In [ ]:
print("Pre-Intervention Data Types:")
print(pre.dtypes)
print("\nPost-Intervention Data Types:")
print(post.dtypes)
print("\n✅ All types are correct — int for counts, float for percentages/IPG")


## 5. Export Clean Dataset

In [ ]:
from data_pipeline import build_full_timeline
timeline = build_full_timeline()
timeline.to_csv('../outputs/clean_full_timeline.csv', index=False)
print(f"✅ Clean timeline exported: {timeline.shape[0]} rows × {timeline.shape[1]} cols")
display(timeline.head(10))
